In [25]:
import pandas as pd
df=pd.read_csv("data/SMSSpamCollection",sep="\t",header=None,names=["label","text"])
df["y"]=(df["label"]=="spam").astype(int)
print(df.shape)
print(df["label"].value_counts())
print(df["y"].mean()*100,"% spam")
df.sample(5,random_state=0)

(5572, 3)
label
ham     4825
spam     747
Name: count, dtype: int64
13.406317300789663 % spam


,label,text,y
4456,ham,"Storming msg: Wen u lift d phne, u say ""HELLO""...",0
690,spam,<Forwarded from 448712404000>Please CALL 08712...,1
944,ham,And also I've sorta blown him off a couple tim...,0
3768,ham,"Sir Goodmorning, Once free call me.",0
1189,ham,All will come alive.better correct any good lo...,0


In [26]:
from sklearn.model_selection import train_test_split
X=df["text"]
y=df["y"]
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=0.4,stratify=y,random_state=42)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42)
for name,part in [("train",y_train),("val",y_val),("test",y_test)]:
    print(name,len(part),round(part.mean()*100,2),"% spam")

train 3343 13.4 % spam
val 1114 13.38 % spam
test 1115 13.45 % spam


In [27]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
dummy=DummyClassifier(strategy="most_frequent")
dummy.fit(X_train,y_train)
pred=dummy.predict(X_val)
print("accuracy:",accuracy_score(y_val,pred))
print("precision:",precision_score(y_val,pred,zero_division=0))
print("f1:",f1_score(y_val,pred))
print(confusion_matrix(y_val,pred))

accuracy: 0.8662477558348295
precision: 0.0
f1: 0.0
[[965   0]
 [149   0]]


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe=Pipeline([
("tfidv",TfidfVectorizer(max_features=5000)),
("clf",LogisticRegression(max_iter=1000))

])
pipe.fit(X_train,y_train)
pred=pipe.predict(X_val)
print("accuracy:",accuracy_score(y_val,pred))
print("precision:",precision_score(y_val,pred,zero_division=0))
print("f1:",f1_score(y_val,pred))
print(confusion_matrix(y_val,pred))

accuracy: 0.9676840215439856
precision: 0.991304347826087
f1: 0.8636363636363636
[[964   1]
 [ 35 114]]


In [29]:
proba=pipe.predict_proba(X_val)[:,1]
print(proba[:10].round(3))
print("recall:",recall_score(y_val,pred))


[0.065 0.052 0.07  0.123 0.039 0.023 0.745 0.014 0.034 0.061]
recall: 0.7651006711409396


In [30]:
for t in [0.5,0.4,0.3,0.2]:
    pred_t=(proba>=t).astype(int)
    print(t,precision_score(y_val,pred_t,zero_division=0),recall_score(y_val,pred_t),f1_score(y_val,pred_t))

0.5 0.991304347826087 0.7651006711409396 0.8636363636363636
0.4 0.992 0.8322147651006712 0.9051094890510949
0.3 0.9485294117647058 0.8657718120805369 0.9052631578947369
0.2 0.85625 0.9194630872483222 0.8867313915857605


In [31]:
for t in [0.5,0.45,0.4,0.35,0.3]:
    pred_t=(proba>=t).astype(int)
    print(t,precision_score(y_val,pred_t,zero_division=0),recall_score(y_val,pred_t),f1_score(y_val,pred_t))

0.5 0.991304347826087 0.7651006711409396 0.8636363636363636
0.45 0.9916666666666667 0.7986577181208053 0.8847583643122676
0.4 0.992 0.8322147651006712 0.9051094890510949
0.35 0.9770992366412213 0.8590604026845637 0.9142857142857143
0.3 0.9485294117647058 0.8657718120805369 0.9052631578947369


In [32]:
import tensorflow as tf
print(tf.__version__)


2.21.0


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=5000)
Xtr=tfidf.fit_transform(X_train).toarray().astype("float32")
Xv=tfidf.transform(X_val).toarray().astype("float32")
print(Xtr.shape,Xv.shape)


(3343, 5000) (1114, 5000)


In [34]:
import tensorflow as tf 
from tensorflow import keras

tf.random.set_seed(42)
model=keras.Sequential([
    keras.layers.Input(shape=(5000,)),
    keras.layers.Dense(32,activation="relu"),
    keras.layers.Dense(1,activation="sigmoid")
])
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history=model.fit(Xtr,y_train,epochs=10,batch_size=32,validation_data=(Xv,y_val),verbose=2)

Epoch 1/10
105/105 - 1s - 12ms/step - accuracy: 0.8648 - loss: 0.4944 - val_accuracy: 0.8905 - val_loss: 0.3150
Epoch 2/10
105/105 - 0s - 4ms/step - accuracy: 0.9315 - loss: 0.2205 - val_accuracy: 0.9587 - val_loss: 0.1624
Epoch 3/10
105/105 - 0s - 4ms/step - accuracy: 0.9782 - loss: 0.1089 - val_accuracy: 0.9785 - val_loss: 0.1071
Epoch 4/10
105/105 - 0s - 4ms/step - accuracy: 0.9901 - loss: 0.0638 - val_accuracy: 0.9811 - val_loss: 0.0854
Epoch 5/10
105/105 - 0s - 4ms/step - accuracy: 0.9937 - loss: 0.0427 - val_accuracy: 0.9829 - val_loss: 0.0753
Epoch 6/10
105/105 - 0s - 5ms/step - accuracy: 0.9952 - loss: 0.0307 - val_accuracy: 0.9829 - val_loss: 0.0698
Epoch 7/10
105/105 - 0s - 4ms/step - accuracy: 0.9964 - loss: 0.0227 - val_accuracy: 0.9829 - val_loss: 0.0669
Epoch 8/10
105/105 - 0s - 4ms/step - accuracy: 0.9973 - loss: 0.0172 - val_accuracy: 0.9829 - val_loss: 0.0655
Epoch 9/10
105/105 - 0s - 4ms/step - accuracy: 0.9979 - loss: 0.0134 - val_accuracy: 0.9829 - val_loss: 0.0651


In [35]:
proba_k=model.predict(Xv).ravel()
for t in [0.5,0.4,0.3]:
    pred_t=(proba_k>=t).astype(int)
    print(t,round(precision_score(y_val,pred_t,zero_division=0),3),round(recall_score(y_val,pred_t),3),round(f1_score(y_val,pred_t),3))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
0.5 0.992 0.872 0.929
0.4 0.985 0.886 0.933
0.3 0.978 0.893 0.933


In [36]:
for t in [0.3,0.25,0.2,0.15]:
    pred_t=(proba_k>=t).astype(int)
    print(t,round(precision_score(y_val,pred_t,zero_division=0),3),round(recall_score(y_val,pred_t),3),round(f1_score(y_val,pred_t),3))

0.3 0.978 0.893 0.933
0.25 0.971 0.899 0.934
0.2 0.938 0.919 0.929
0.15 0.908 0.926 0.917


In [37]:
THRESH_LR=0.35
THRESH_KERAS=0.25
Xte=tfidf.transform(X_test).toarray().astype("float32")
proba_lr_test=pipe.predict_proba(X_test)[:,1]
proba_k_test=model.predict(Xte).ravel()
for name,p,t in [("LogReg",proba_lr_test,THRESH_LR),("Keras",proba_k_test,THRESH_KERAS)]:
    pred_t=(p>=t).astype(int)
    print(name,t,round(precision_score(y_test,pred_t,zero_division=0),3),round(recall_score(y_test,pred_t),3),round(f1_score(y_test,pred_t),3))
    print(confusion_matrix(y_test,pred_t))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
LogReg 0.35 0.971 0.907 0.938
[[961   4]
 [ 14 136]]
Keras 0.25 0.947 0.947 0.947
[[957   8]
 [  8 142]]
